In [1]:
from pathlib import Path
import re
import pandas as pd
import seaborn as sns

In [2]:
WORKLOADS = ["faiss_10M", "lulesh2.0_s400", "DuckDB-TPCH-sf100", "DuckDB-TPCDS-sf100", "mg.D.x", "XSBench", "pr-kron.sg", "pr-twitter.sg", "bc-kron.sg", "bc-twitter.sg"]

In [3]:
def parse_time_value(raw: str) -> float:
    """Convert strings like '7m56.676s' into seconds."""
    
    raw = raw.strip()
    match = re.fullmatch(r"(?:(\d+)m)?([0-9.]+)s", raw)
    if not match:
        raise ValueError(f"Unrecognized time format: {raw}")
    minutes = int(match.group(1) or 0)
    seconds = float(match.group(2))
    return minutes * 60 + seconds

def parse_time_file(path: Path) -> dict:
    metrics = {}
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            key, value = line.split("\t", 1)
            metrics[key] = parse_time_value(value)
    return metrics

def extract_mib(path: Path) -> int | None:
    match = re.search(r"(\d+)MiB", path.name)
    return int(match.group(1)) if match else None


def extract_run(path: Path) -> int | None:
    match = re.search(r"run(\d+)", path.name)
    return int(match.group(1)) if match else None

def parse_log_file(path: Path) -> dict:
    metrics = {}
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line or "=" not in line:
                continue
            key, value = line.split("=", 1)
            metrics[key.strip()] = float(value.strip())
    return metrics

In [12]:
dfs = []

for workload in WORKLOADS:
    roots = {
        "arms": ("6.18", Path(f"/users/zimooo2/tiering_solutions/times/arms/{workload}")),
        "hybridtier": ("6.2+", (Path(f"/users/zimooo2/tiering_solutions/times/hybridtier/{workload}"))),
        "arms-6.2": ("6.2+", Path(f"/users/zimooo2/tiering_solutions/times/arms-6.2/{workload}")),
        "model": ("6.18", Path(f"/users/zimooo2/tiering_solutions/times/model/{workload}")),
    }

    records: list[dict] = []
    for label, (kernel, root) in roots.items():
        for path in sorted(root.glob("*.time")):
            if "train" in path.name:
                continue
            metrics = parse_time_file(path)
            try:
                log_metrics = parse_log_file(Path(f"{path.parent}/max_dram_hugepages_{path.stem}.log"))
            except FileNotFoundError:
                log_metrics = {}
                
                
            
                
            m = re.search(
                r"_discounted_reward_(\d+)_.*_l2-(true|false)_(\d+)_([0-9]*\.?[0-9]+)\.time$",
                path.name,
            )
            parsed = {}
            if m:
                parsed = {
                    "discounted_reward": int(m.group(1)),
                    "minmax": m.group(2) == "true",
                    "hist_length": int(m.group(3)),
                    "penalty": float(m.group(4)),
                }

            records.append({
                "policy": label,
                "kernel": kernel,
                "file": path.name,
                "MiB": extract_mib(path),
                "run": extract_run(path),
                **metrics,
                **log_metrics,
                **parsed,
            })

    df = pd.DataFrame(records)
    df["workload"] = workload

    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df


arms_df = (
    df.loc[df["policy"] == "arms", ["MiB", "workload", "real"]]
    .rename(columns={"real": "real_arms"})
    .drop_duplicates(subset=["MiB", "workload"])
)

df = (
    df.merge(arms_df, on=["MiB", "workload"], how="left")
)

# >1 means faster than arms, <1 means slower than arms
df["relative_score_vs_arms"] = df["real_arms"] / df["real"]
df

,policy,kernel,file,MiB,run,real,user,sys,discounted_reward,minmax,hist_length,penalty,workload,real_arms,relative_score_vs_arms
0,arms,6.18,10000MiB_run1.time,10000,1,483.170,0.010,0.024,NaN,NaN,NaN,NaN,faiss_10M,483.170,1.000000
1,arms,6.18,10000MiB_run2.time,10000,2,491.678,0.013,0.022,NaN,NaN,NaN,NaN,faiss_10M,483.170,0.982696
2,arms,6.18,10000MiB_run3.time,10000,3,504.333,0.013,0.020,NaN,NaN,NaN,NaN,faiss_10M,483.170,0.958038
3,arms,6.18,4000MiB_run1.time,4000,1,536.373,0.012,0.022,NaN,NaN,NaN,NaN,faiss_10M,536.373,1.000000
4,arms,6.18,4000MiB_run2.time,4000,2,532.691,0.010,0.023,NaN,NaN,NaN,NaN,faiss_10M,536.373,1.006912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,model,6.18,8000MiB_run2_model_discounted_reward_95_bc-twi...,8000,2,107.616,0.012,0.024,95.0,False,4.0,0.9,bc-twitter.sg,NaN,NaN
444,model,6.18,8000MiB_run3_model_discounted_reward_95_bc-twi...,8000,3,108.973,0.013,0.021,95.0,False,4.0,0.9,bc-twitter.sg,NaN,NaN
445,model,6.18,9MiB_run9_model_discounted_reward_90_bc-twitte...,9,9,121.836,0.005,0.019,90.0,True,4.0,0.9,bc-twitter.sg,NaN,NaN
446,model,6.18,9MiB_run9_model_discounted_reward_95_bc-twitte...,9,9,113.744,0.014,0.020,95.0,True,4.0,0.9,bc-twitter.sg,NaN,NaN


In [13]:
df[df["MiB"] > 3000].sort_values(["MiB", "workload", "policy"]).to_csv("summary.csv", index=False)